[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch11-hw-acceleration/03-mang-systolic-va-chia-khoi.ipynb)

# Mảng systolic, chia khối và thứ tự vòng lặp: mỗi byte được dùng lại bao nhiêu lần

Sổ tay 01 và 02 cho thấy phần lớn phép toán trong học máy nghẽn ở bộ nhớ, và cách thoát là làm
cho mỗi byte đã nạp phục vụ nhiều phép tính hơn. Sổ tay này đi vào bên trong bộ tăng tốc
(accelerator) để xem điều đó được làm thế nào: bằng cách sắp các phần tử xử lý thành một lưới,
bằng cách chia phép nhân lớn thành khối vừa bộ nhớ nhanh, và bằng thứ tự của các vòng lặp.

Ở đây không có phép đo thời gian nào. Mọi con số đều đếm được, nên mỗi ô đều kiểm được kết quả
của mình một cách chính xác.

**Bạn sẽ làm:**
1. tính lại ví dụ Tensor Core của sách cho một đầu tập trung;
2. mô phỏng từng chu kỳ một mảng systolic giữ trọng số tại chỗ, kiểm kết quả đúng tuyệt đối,
   đếm chu kỳ nạp đầy và xả, rồi tính lại napkin math 1.1;
3. chia khối một phép nhân ma trận, đếm lưu lượng, và xem bộ nhớ nháp đặt trần cho cỡ khối;
4. chạy cả 120 thứ tự vòng lặp của ví dụ 1.4 trên một tệp thanh ghi có sức chứa thay đổi.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Tensor Core: cùng số phép, tổ chức theo khối

Ví dụ của sách: một đầu tập trung tính $QK^T$ cho một chuỗi 2,048 token, embedding 64 chiều. Đó là
phép nhân một ma trận $2048 \times 64$ với một ma trận $64 \times 2048$. Mỗi phần tử của kết quả
$2048 \times 2048$ là một tích vô hướng 64 số hạng.

**Dự đoán:** phép nhân này có khoảng bao nhiêu triệu phép nhân cộng dồn (MAC)?

In [ ]:
token = sach(2048, nguon="2,048 · 64")
chieu = sach(64, nguon="2,048 · 64")
mac = token * token * chieu
theo_sach("MAC của QKᵀ (triệu)", mac / 1e6, sach=268.4, nguon="268.4 · 536.9")
flop = 2 * mac                        # nhân và cộng đếm riêng
theo_sach("FLOP (triệu)", flop / 1e6, sach=536.9, nguon="268.4 · 536.9")

GHz = sach(2, nguon="2 · 268.4")      # bộ xử lý vô hướng: một FLOP mỗi chu kỳ ở 2 GHz
t_vo_huong = flop / (GHz * 1e9)
theo_sach("thời gian vô hướng (ms)", t_vo_huong * 1e3, sach=268.4, nguon="2 · 268.4")
t_khoi = sach(0.5, nguon="0.5 · 536.9")   # ms, giả định minh hoạ của sách cho đường chia khối
theo_sach("nhanh hơn, theo giả định minh hoạ", t_vo_huong * 1e3 / t_khoi, sach=536.9, nguon="0.5 · 536.9")

Hai số 268.4 trùng nhau không phải ngẫu nhiên: một FLOP mỗi chu kỳ ở 2 GHz là đúng một MAC mỗi
nano giây, nên số mili giây bằng số triệu MAC. Còn 0.5 ms là giả định minh hoạ của sách, không
phải số đo. Điều sách muốn nói là tốc độ ấy không đến từ xung nhịp nhanh hơn, mà từ việc mỗi lệnh
xử lý cả một khối ma trận. Phần tiếp theo xem một cấu trúc làm được như vậy.

## 2. Một mảng systolic, từng chu kỳ một

Mảng systolic (systolic array) là một lưới phần tử xử lý (processing element, PE), mỗi phần tử
làm một phép nhân cộng dồn mỗi chu kỳ và chỉ nói chuyện với hàng xóm. Ta mô phỏng cách giữ trọng
số tại chỗ (weight stationary) mà sách mô tả cho phép nhân $C = A \cdot B$:

* mảng có $K$ hàng và $N$ cột; phần tử ở hàng $k$, cột $j$ giữ sẵn $B_{kj}$ và không bao giờ nạp lại;
* hàng $m$ của $A$ đi vào từ mép trái, lệch nhau theo thời gian: $A_{mk}$ vào hàng $k$ ở chu kỳ
  $m + k$, rồi mỗi chu kỳ sang phải một ô;
* tổng riêng phần đi xuống theo cột: mỗi phần tử cộng tích của mình vào tổng từ ô phía trên rồi
  đẩy xuống, nên $C_{mj}$ rời đáy cột $j$ ở chu kỳ $m + K - 1 + j$.

Độ lệch ấy là để hai toán hạng của cùng một tích gặp nhau đúng ô, đúng lúc. Mô phỏng dưới đây chỉ
dùng các thanh ghi giữa các ô, không bao giờ tính `A @ B`, rồi so kết quả với `A @ B` trên số
nguyên, nên phải bằng nhau tuyệt đối.

In [ ]:
def mang_systolic(A, B, ghi_lai=False):
    """Mảng systolic K × N giữ B tại chỗ; A đi từ trái sang, tổng riêng phần đi từ trên xuống.

    Trả về C, số chu kỳ, số MAC đã làm, số lần chạm bộ nhớ ở mép mảng (nạp B, đưa A vào, ghi C),
    và nếu ghi_lai, ảnh chụp từng chu kỳ: mỗi ô đang phục vụ hàng nào của A (-1 là nhàn rỗi).
    """
    M, K = A.shape
    N = B.shape[1]
    a_reg = np.zeros((K, N), A.dtype)        # giá trị A đang nằm ở mỗi ô
    p_reg = np.zeros((K, N), A.dtype)        # tổng riêng phần mỗi ô vừa đẩy xuống
    C = np.zeros((M, N), A.dtype)
    hang, cot = np.arange(K), np.arange(N)
    so_chu_ky = M + K + N - 2
    so_mac, a_vao, c_ra, anh = 0, 0, 0, []
    for c in range(so_chu_ky):
        a_moi = np.empty_like(a_reg)
        a_moi[:, 1:] = a_reg[:, :-1]                         # sang phải một ô
        m_vao = c - hang                                     # hàng của A vào mép trái ở chu kỳ c
        hop_le = (m_vao >= 0) & (m_vao < M)
        a_moi[:, 0] = np.where(hop_le, A[np.clip(m_vao, 0, M - 1), hang], 0)
        a_vao += int(hop_le.sum())
        p_moi = np.empty_like(p_reg)
        p_moi[0] = a_moi[0] * B[0]                           # hàng trên cùng bắt đầu từ 0
        p_moi[1:] = p_reg[:-1] + a_moi[1:] * B[1:]           # cộng vào tổng từ ô phía trên
        m_o = c - hang[:, None] - cot[None, :]               # mỗi ô đang phục vụ hàng nào của A
        dang_lam = (m_o >= 0) & (m_o < M)
        so_mac += int(dang_lam.sum())
        if ghi_lai:
            anh.append(np.where(dang_lam, m_o, -1))
        m_ra = c - (K - 1) - cot                             # hàng của C rời đáy mỗi cột
        ra = (m_ra >= 0) & (m_ra < M)
        C[m_ra[ra], cot[ra]] = p_moi[K - 1, ra]
        c_ra += int(ra.sum())
        a_reg, p_reg = a_moi, p_moi
    cham = {"nạp B": K * N, "đưa A vào": a_vao, "ghi C": c_ra}
    return C, so_chu_ky, so_mac, cham, anh


rng = np.random.default_rng(0)
for M, K, N in [(2, 2, 2), (4, 4, 4), (6, 6, 6), (5, 3, 7), (300, 128, 128)]:
    A = rng.integers(-9, 10, (M, K))
    B = rng.integers(-9, 10, (K, N))
    C, chu_ky, so_mac, cham, _ = mang_systolic(A, B)
    assert np.array_equal(C, A @ B)
    assert chu_ky == M + K + N - 2 and so_mac == M * K * N
    assert cham == {"nạp B": K * N, "đưa A vào": M * K, "ghi C": M * N}
    print(f"A {M}×{K}, B {K}×{N}: đúng tuyệt đối, {chu_ky} chu kỳ, {so_mac:,} MAC, chạm mép mảng {sum(cham.values()):,} lần")

Với tích $2 \times 2$ của sách, ta in ra từng tích được làm ở đâu và lúc nào. Sách viết $C_{00}$
gộp $A_{00}B_{00}$ với $A_{01}B_{10}$: hai tích ấy rơi vào hai ô khác nhau của cột 0, cách nhau
một chu kỳ, đúng lúc tổng riêng phần từ ô trên đi xuống tới ô dưới.

In [ ]:
n2 = sach(2, nguon="2 nhân 2 · 16,384 · 1,200 GB/s")
A2 = np.array([[1, 2], [3, 4]])
B2 = np.array([[5, 6], [7, 8]])
C2, chu_ky2, _, _, anh2 = mang_systolic(A2, B2, ghi_lai=True)
for c, a in enumerate(anh2):
    viec = [f"PE({k},{j}): A{m}{k}·B{k}{j}" for k in range(n2) for j in range(n2) if (m := a[k, j]) >= 0]
    print(f"chu kỳ {c}: " + ", ".join(viec))
assert np.array_equal(C2, A2 @ B2)
print("C =", C2.tolist())

Hình dưới chụp một lưới 6 × 6 ở bốn chu kỳ, với 6 hàng của $A$ đi qua. Mỗi ô ghi hàng nào của
$A$ nó đang phục vụ; ô trắng là nhàn rỗi. Công việc lan qua lưới thành những đường chéo: lúc đầu
chỉ góc trên trái làm việc (nạp đầy), rồi một dải chéo rộng nhất ở giữa, rồi chỉ góc dưới phải (xả).
Với chỉ 6 hàng đi qua, lưới không lúc nào bận hết.

In [ ]:
from matplotlib.colors import ListedColormap

M6 = K6 = N6 = 6
_, chu_ky6, _, _, anh6 = mang_systolic(np.ones((M6, K6), int), np.ones((K6, N6), int), ghi_lai=True)
mau = ListedColormap(["white", "#cfe1f7", "#a4c7f0", "#74a8e6", "#4a8ad9", "#2a6fc4", "#1d4f91"])
fig, axs = plt.subplots(1, 4, figsize=(10, 2.9))
for ax, c in zip(axs, (2, 5, 9, 13)):
    a = anh6[c]
    ax.imshow(a + 1, cmap=mau, vmin=0, vmax=M6, origin="upper")
    for k in range(K6):
        for j in range(N6):
            if a[k, j] >= 0:
                ax.text(j, k, str(a[k, j]), ha="center", va="center", fontsize=7,
                        color="white" if a[k, j] >= 3 else "black")
    ax.set_xticks(np.arange(-0.5, N6), minor=True)
    ax.set_yticks(np.arange(-0.5, K6), minor=True)
    ax.grid(which="minor", color="0.75", lw=0.6)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    ax.set_title(f"chu kỳ {c}: {int((a >= 0).sum())}/36 ô làm việc", fontsize=9)
fig.suptitle("Lưới 6 × 6 minh hoạ: số trong ô là hàng của A mà ô đang phục vụ", fontsize=10)
plt.tight_layout()
plt.show()

### Nạp đầy và xả

Một khối đứng riêng cần $M + K + N - 2$ chu kỳ cho $MKN$ phép MAC, trên một lưới có $KN$ ô. Vậy mức
sử dụng (utilization) của lưới là

$$\frac{MKN}{(M + K + N - 2)\,KN} = \frac{M}{M + K + N - 2}.$$

Sách nói các chu kỳ nạp đầy và xả vẫn là một phần chi phí. Công thức cho biết phần ấy lớn tới đâu:
nó chỉ nhỏ khi số hàng $M$ chảy qua một lần nạp trọng số lớn hơn nhiều so với cạnh của lưới. Mô
phỏng ở đây xử lý một khối đứng riêng; nếu trọng số của khối sau được nạp trong lúc khối trước
đang xả, phần hao này nhỏ đi.

**Dự đoán:** trên lưới 128 × 128 của sách, cần bao nhiêu hàng $M$ để lưới bận ít nhất 90 % thời gian?

In [ ]:
canh = sach(128, nguon="128 · 128×128")
theo_sach("MAC mỗi chu kỳ của lưới 128 × 128", canh * canh, sach=16_384, nguon="16,384")


def muc_su_dung(M, K, N):
    return M / (M + K + N - 2)


# Mô phỏng thật trên lưới 128 × 128 phải ra đúng công thức.
for M in (1, 16, 128, 512):
    _, chu_ky, so_mac, _, _ = mang_systolic(rng.integers(-3, 4, (M, canh)), rng.integers(-3, 4, (canh, canh)))
    assert so_mac * (M + 2 * canh - 2) == M * chu_ky * canh * canh     # MKN / (chu kỳ · KN) = M / (M + K + N − 2)
    print(f"M = {M:>3}: mức sử dụng {muc_su_dung(M, canh, canh):.1%} (mô phỏng và công thức trùng nhau)")
M_90 = next(M for M in range(1, 10_000) if muc_su_dung(M, canh, canh) >= 0.9)
print(f"cần ít nhất M = {M_90:,} hàng để lưới 128 × 128 bận 90 %")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
for M, mau_ve in ((6, "#2a78d6"), (30, "#eb6834")):
    _, chu_ky, _, _, anh = mang_systolic(np.ones((M, 6), int), np.ones((6, 6), int), ghi_lai=True)
    ax1.plot(range(chu_ky), [(a >= 0).mean() for a in anh], "o-", color=mau_ve, ms=3.5,
             label=f"M = {M} hàng của A")
ax1.set_ylim(0, 1.25)
ax1.set_yticks([0, 0.25, 0.5, 0.75, 1], ["0", "25 %", "50 %", "75 %", "100 %"])
ax1.set_xlabel("chu kỳ")
ax1.set_ylabel("phần lưới đang làm việc")
ax1.set_title("Lưới 6 × 6: nạp đầy, chạy, xả")
ax1.legend(loc="upper right", fontsize=8, frameon=False, ncol=2)

cac_M = np.unique(np.logspace(0, 4, 200).astype(int))
ax2.semilogx(cac_M, [100 * muc_su_dung(M, canh, canh) for M in cac_M], color="#2a78d6", lw=2)
for M, cho_chu in ((128, (0.62, 0.30)), (M_90, (0.62, 0.52)), (4096, (0.62, 0.41))):
    y = 100 * muc_su_dung(M, canh, canh)
    ax2.plot(M, y, "o", color="#2a78d6", ms=6, mec="white", mew=1.2)
    ax2.annotate(f"M = {M:,}: {muc_su_dung(M, canh, canh):.0%}", (M, y), xytext=cho_chu,
                 textcoords="axes fraction", fontsize=8, ha="left",
                 arrowprops=dict(arrowstyle="-", color="0.55", lw=0.8, shrinkB=4))
ax2.set_ylim(0, 105)
ax2.set_xlabel("số hàng M chảy qua một lần nạp trọng số")
ax2.set_ylabel("mức sử dụng (%)")
ax2.set_title("Lưới 128 × 128: M ÷ (M + K + N − 2)")
plt.tight_layout()
plt.show()

Ở hình trái, với 6 hàng, lưới chưa kịp đầy đã bắt đầu xả. Với 30 hàng, lưới có một đoạn bận gần
trọn, nhưng hai đầu vẫn là hai cái dốc. Ở hình phải, một lưới 128 × 128 chỉ cho 128 hàng chảy qua
thì bận chưa tới một nửa thời gian. Đây là lý do batch nhỏ khó lấp đầy một mảng lớn, và cũng là
một dạng của lời cảnh báo trong chương: hiệu năng đỉnh của một mảng giả định mọi ô đều làm việc.

### Napkin math 1.1: năng lượng của việc dùng lại

Sách so mảng systolic với một mốc ngây thơ cố ý đơn giản: mỗi phép nhân cộng dồn đọc ba toán hạng
và ghi một kết quả, tất cả qua DRAM, không có cache hay thanh ghi nào dùng lại. Mỗi lần chạm DRAM
tốn khoảng 640 pJ, phép tính tốn 1 pJ. Mảng 128 × 128 thì chỉ nạp A và B một lần ở mép, nên sách
đếm 2 lần nạp cho mỗi 128 phép.

In [ ]:
dram_pJ = sach(640, nguon="4 · 640 · 1 · 2561")
tinh_pJ = sach(1, nguon="4 · 640 · 1 · 2561")
lan_ngay_tho = sach(4, nguon="4 · 640 · 1 · 2561")       # 3 lần đọc + 1 lần ghi
E_ngay_tho = lan_ngay_tho * dram_pJ + tinh_pJ
theo_sach("mốc ngây thơ (pJ mỗi phép)", E_ngay_tho, sach=2561, nguon="4 · 640 · 1 · 2561")

lan_systolic = sach(2, nguon="2 · 0.016 · 11") / canh     # 2 lần nạp mỗi 128 phép
theo_sach("mảng systolic: lần chạm DRAM mỗi phép", lan_systolic, sach=0.016, nguon="2 · 0.016 · 11")
E_systolic = lan_systolic * dram_pJ + tinh_pJ
theo_sach("mảng systolic (pJ mỗi phép)", E_systolic, sach=11, nguon="2 · 0.016 · 11")
theo_sach("lợi thế năng lượng", E_ngay_tho / E_systolic, sach=232.8, nguon="232.8")
print(f"nếu dùng đúng số in 0.016: {0.016 * dram_pJ + tinh_pJ:.2f} pJ và {E_ngay_tho / (0.016 * dram_pJ + tinh_pJ):.1f} lần;"
      " sách dùng 2/128 = 0.015625 rồi làm tròn khi in")

Bộ mô phỏng của ta đếm được một con số khác, vì nó đếm cả lượt ghi kết quả ra. Với $M = K = N =
128$, mép mảng bị chạm $MK + KN + MN = 3 \cdot 128^2$ lần cho $128^3$ phép, tức 3 lần mỗi 128 phép
thay vì 2. Cả hai cách đếm đều đúng với giả định của chúng; điều đáng nhớ là sách tự nói tỉ số này
đo **tính cục bộ**, không phải một khoảng cách cố hữu giữa phép tính vectơ và phép tính systolic,
và cache, chia khối thanh ghi hay mảng không được dùng hết đều làm nó thay đổi nhiều.

In [ ]:
_, _, so_mac128, cham128, _ = mang_systolic(rng.integers(-3, 4, (canh, canh)), rng.integers(-3, 4, (canh, canh)))
assert sum(cham128.values()) * canh == 3 * so_mac128                 # 3 lần chạm mỗi 128 phép
lan_mo_phong = sum(cham128.values()) / so_mac128
E_mo_phong = lan_mo_phong * dram_pJ + tinh_pJ
print(f"bộ mô phỏng: {lan_mo_phong:.4f} lần chạm mỗi phép, {E_mo_phong:.0f} pJ, lợi thế {E_ngay_tho / E_mo_phong:.1f} lần")
print(f"chỉ đếm phần nạp như sách: {(cham128['nạp B'] + cham128['đưa A vào']) / so_mac128:.6f} lần chạm mỗi phép")

## 3. Chia khối: khi phép nhân lớn hơn lưới

Đồ thị tính toán thấy một phép nhân $4096 \times 4096$; silicon chỉ có một lưới 128 × 128. Trình
biên dịch bắc cầu bằng chia khối (tiling): cắt kết quả thành các khối vừa lưới, và mỗi khối kết quả
cộng dồn qua các khối của chiều rút gọn.

In [ ]:
n_lop = sach(4096, nguon="4,096 · 128 · 1,024 · 32 · 32,768")
T_luoi = sach(128, nguon="4,096 · 128 · 1,024 · 32 · 32,768")
khoi_ra = (n_lop // T_luoi) ** 2
khoi_rut_gon = n_lop // T_luoi
theo_sach("số khối đầu ra", khoi_ra, sach=1024, nguon="4,096 · 128 · 1,024 · 32 · 32,768")
theo_sach("số khối rút gọn mỗi khối đầu ra", khoi_rut_gon, sach=32, nguon="4,096 · 128 · 1,024 · 32 · 32,768")
theo_sach("số tích khối A × khối B", khoi_ra * khoi_rut_gon, sach=32_768, nguon="4,096 · 128 · 1,024 · 32 · 32,768")
theo_sach("mỗi giá trị A nạp vào được dùng cho bao nhiêu cột", T_luoi, sach=128, nguon="128 lần · 129 · 127")

rong = sach(129, nguon="128 lần · 129 · 127")
so_khoi = math.ceil(rong / T_luoi)
theo_sach("rộng 129: số cột nhàn rỗi ở khối thứ hai", so_khoi * T_luoi - rong, sach=127, nguon="128 lần · 129 · 127")
print(f"rộng {rong} cần {so_khoi} khối, mức sử dụng theo chiều rộng {rong / (so_khoi * T_luoi):.1%}")

In [ ]:
cac_rong = np.arange(1, 641)
su_dung = cac_rong / (T_luoi * np.ceil(cac_rong / T_luoi))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(cac_rong, 100 * su_dung, color="#2a78d6", lw=1.6)
for w, (dx, dy), ha in ((128, (-6, 6), "right"), (129, (8, -2), "left"), (384, (-6, 6), "right"), (385, (8, -2), "left")):
    ax.plot(w, 100 * w / (T_luoi * math.ceil(w / T_luoi)), "o", color="#eb6834", ms=5, mec="white", mew=1)
    ax.annotate(f"rộng {w}: {w / (T_luoi * math.ceil(w / T_luoi)):.0%}", (w, 100 * w / (T_luoi * math.ceil(w / T_luoi))),
                xytext=(dx, dy), textcoords="offset points", fontsize=8, ha=ha)
ax.set_xlim(0, 640)
ax.set_xticks(range(0, 641, 128))
ax.set_ylim(0, 112)
ax.set_xlabel("chiều rộng của tầng")
ax.set_ylabel("mức sử dụng theo chiều rộng (%)")
ax.set_title("Thuế phần rìa trên lưới rộng 128")
plt.tight_layout()
plt.show()

Mỗi khi chiều rộng vượt một bội của 128 một chút, mức sử dụng tụt xuống: rộng 129 cần hai khối,
và khối thứ hai chỉ dùng 1 trong 128 cột. Răng cưa này nhỏ dần khi tầng rộng ra, vì phần rìa chỉ
là một khối trong nhiều khối. Sách gọi đó là thuế phần rìa trả bằng silicon không được dùng hết.

### Đếm lưu lượng của một phép nhân chia khối

Hàm dưới nhân hai ma trận $n \times n$ theo khối $T \times T$, đúng vòng lặp mà sách mô tả: nạp một
khối A và một khối B vào bộ nhớ nhanh, cộng dồn tích của chúng vào một khối C nằm yên trong bộ nhớ
nhanh, xong cả chiều rút gọn mới ghi khối C ra. Hàm đếm mọi phần tử đi qua ranh giới giữa bộ nhớ
chậm và bộ nhớ nhanh.

Mỗi khối C cần $n/T$ cặp khối A, B, mỗi cặp $2T^2$ phần tử, và có $(n/T)^2$ khối C. Tổng lượt nạp là
$2n^3/T$, cộng $n^2$ lượt ghi. Vậy mật độ tính toán ở FP32 là

$$I(T) = \frac{2n^3}{4\,(2n^3/T + n^2)} \approx \frac{T}{4},$$

còn bộ nhớ nhanh phải chứa cùng lúc ba khối, $3T^2$ phần tử. Đó đúng là câu của sách: khối lớn hơn
dùng lại mỗi byte cho nhiều phép hơn, và trần của nó là lượng A, B, C vừa bộ nhớ trên chip.

**Dự đoán:** nếu bỏ chia khối, tức $T = 1$, mật độ còn bao nhiêu FLOP/byte?

In [ ]:
def nhan_chia_khoi(A, B, T):
    """C = A·B theo khối T × T; đếm phần tử nạp từ bộ nhớ chậm và ghi ra bộ nhớ chậm."""
    n = A.shape[0]
    C = np.zeros((n, n), A.dtype)
    nap = ghi = 0
    for i in range(0, n, T):
        for j in range(0, n, T):
            khoi_C = np.zeros((T, T), A.dtype)              # nằm yên trong bộ nhớ nhanh
            for k in range(0, n, T):
                khoi_A, khoi_B = A[i:i + T, k:k + T], B[k:k + T, j:j + T]
                nap += khoi_A.size + khoi_B.size            # hai khối đi vào bộ nhớ nhanh
                khoi_C += khoi_A @ khoi_B
            C[i:i + T, j:j + T] = khoi_C
            ghi += khoi_C.size                              # ghi ra một lần
    return C, nap, ghi


n = 256
A = rng.integers(-5, 6, (n, n))
B = rng.integers(-5, 6, (n, n))
C_chuan = A @ B
for T in (8, 16, 32, 64, 128, 256):
    C, nap, ghi = nhan_chia_khoi(A, B, T)
    assert np.array_equal(C, C_chuan)
    assert nap == 2 * n**3 // T and ghi == n * n
    print(f"T = {T:>3}: đúng tuyệt đối, nạp {nap:>10,} phần tử, ghi {ghi:,}")


def mat_do_khoi(n, T, byte=4):
    return 2 * n**3 / (byte * (2 * n**3 / T + n**2))


print(f"\nn = 4,096 ở FP32: T = 1 cho {mat_do_khoi(4096, 1):.3f} FLOP/byte, T = 128 cho {mat_do_khoi(4096, 128):.1f}")

### Khối nào vừa bộ nhớ nháp?

Phần này phỏng theo phần B của phòng thí nghiệm đi kèm chương. Một bộ tăng tốc giữ các khối trong
bộ nhớ nháp (scratchpad memory), một bộ nhớ trên chip do phần mềm quản lý. Sức chứa dưới đây là
**giả định của sổ tay**, không phải số của sách hay của một chip cụ thể; bạn có thể đổi nó.

Phòng thí nghiệm còn đặt một câu hỏi thứ hai: nếu kết quả của phép nhân còn đi qua một phép theo
từng phần tử (chẳng hạn ReLU) ở một kernel riêng, thì $C$ phải được ghi ra rồi đọc lại, thêm $2n^2$
phần tử lưu lượng. Hợp nhất phép ấy vào phép nhân thì phần này mất đi.

In [ ]:
BO_NHO_NHAP = 256 * 1024          # byte; giả định của sổ tay
n = 4096
print(f"bộ nhớ nháp giả định: {BO_NHO_NHAP // 1024} KiB; phép nhân {n:,} × {n:,} ở FP32\n")
print(f"{'T':>4} {'cần (KiB)':>10} {'vừa?':>6} {'lưu lượng, hợp nhất (MB)':>25} {'không hợp nhất (MB)':>20} {'I (FLOP/byte)':>14}")
bang = []
for T in (16, 32, 64, 128, 256, 512):
    can = 3 * T * T * 4
    vua = can <= BO_NHO_NHAP
    hop_nhat = 4 * (2 * n**3 / T + n * n)
    khong_hop_nhat = hop_nhat + 4 * 2 * n * n
    bang.append((T, can, vua, mat_do_khoi(n, T)))
    print(f"{T:>4} {can / 1024:>10.0f} {'vừa' if vua else 'tràn':>6} {hop_nhat / 1e6:>25,.0f} {khong_hop_nhat / 1e6:>20,.0f}"
          f" {mat_do_khoi(n, T):>14.1f}")
T_lon_nhat = max(T for T, _, vua, _ in bang if vua)
print(f"\nkhối lớn nhất vừa bộ nhớ nháp giả định: T = {T_lon_nhat}")

In [ ]:
cac_T = 2 ** np.arange(0, 10)
can_KiB = 3 * cac_T.astype(float) ** 2 * 4 / 1024
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(can_KiB, [mat_do_khoi(4096, T) for T in cac_T], "o-", color="#2a78d6", ms=6, mec="white", mew=1.2)
for T, kib in zip(cac_T, can_KiB):
    tren_trai = kib < BO_NHO_NHAP / 1024
    ax.annotate(f"T = {T}", (kib, mat_do_khoi(4096, T)), xytext=(-6, 7) if tren_trai else (8, -12),
                textcoords="offset points", fontsize=8, ha="right" if tren_trai else "left")
ax.axvline(BO_NHO_NHAP / 1024, color="#eb6834", lw=1.4, ls="--")
ax.text(BO_NHO_NHAP / 1024 * 1.15, 0.12, "bộ nhớ nháp\ngiả định", fontsize=8, color="#b5461c", va="bottom")
ax.axvspan(BO_NHO_NHAP / 1024, can_KiB[-1] * 3, color="#eb6834", alpha=0.08, lw=0)
ax.set_xlim(can_KiB[0] / 3, can_KiB[-1] * 3)
ax.set_ylim(0.1, 400)
ax.set_xlabel("bộ nhớ nhanh cần cho ba khối (KiB, thang log)")
ax.set_ylabel("mật độ tính toán (FLOP/byte, FP32)")
ax.set_title("Phép nhân 4,096 × 4,096: khối càng lớn, mỗi byte càng được dùng lại")
plt.tight_layout()
plt.show()

Mỗi lần gấp đôi cạnh khối, mật độ tăng gần gấp đôi nhưng bộ nhớ nhanh cần tăng gấp bốn. Vùng
tô nhạt bên phải là những khối không vừa: ở đó phép nhân phải chia lại, không thể chạy chậm đi mà
vẫn đúng. Trong bảng, phần không hợp nhất chỉ thêm một ít vào lưu lượng của phép nhân, vì phép nhân
đã có mật độ cao. Với một chuỗi toàn phép theo từng phần tử như ở sổ tay 02, chính các lượt đi về
ấy là gần hết lưu lượng.

## 4. Ví dụ 1.4: cùng một tích chập, 120 thứ tự vòng lặp

Sách lấy một tích chập 16 bộ lọc $3 \times 3$ trên đầu vào $8 \times 8$ một kênh. Nó là năm vòng lặp
lồng nhau: hàng và cột của đầu ra ($H$, $W$), bộ lọc ($C$), hàng và cột của bộ lọc ($F_h$, $F_w$).
Mọi thứ tự của năm vòng ấy cho cùng kết quả, nhưng mỗi thứ tự đòi bộ nhớ một kiểu khác.

* **Thứ tự A, giữ trọng số tại chỗ:** vòng bộ lọc ở ngoài, vòng không gian ở trong. Mỗi trọng số
  nạp một lần rồi dùng cho cả 36 vị trí đầu ra.
* **Thứ tự B, giữ đầu ra tại chỗ (output stationary):** vòng không gian ở ngoài, vòng bộ lọc ở trong. Mỗi vị trí đầu ra
  cần cả 16 bộ lọc; nếu tệp thanh ghi (register file) không chứa hết, trọng số bị nạp lại.

In [ ]:
bo_loc = sach(16, nguon="16 · 3 × 3 · 8 × 8 · 5 · 120")
F = sach(3, nguon="16 · 3 × 3 · 8 × 8 · 5 · 120")
canh_vao = sach(8, nguon="16 · 3 × 3 · 8 × 8 · 5 · 120")
so_vong = sach(5, nguon="16 · 3 × 3 · 8 × 8 · 5 · 120")
canh_ra = canh_vao - F + 1
theo_sach("số vị trí đầu ra", canh_ra * canh_ra, sach=36, nguon="36 · 16 × 9 · 144 · 36 × 144 · 36 lần")
theo_sach("số trọng số", bo_loc * F * F, sach=144, nguon="36 · 16 × 9 · 144 · 36 × 144 · 36 lần")
theo_sach("số thứ tự vòng lặp, 5!", math.factorial(so_vong), sach=120, nguon="16 · 3 × 3 · 8 × 8 · 5 · 120")

Ta chạy tích chập theo từng thứ tự trong cả 120 thứ tự, kiểm kết quả với một cách tính trực tiếp,
và ghi lại chuỗi trọng số được đọc. Rồi ta đưa chuỗi ấy qua một tệp thanh ghi chứa được $R$ trọng
số, thay ra trọng số lâu nhất chưa dùng (LRU). Mỗi lần đọc trượt là một lần nạp.

Có một mẹo để không phải chạy lại cho từng $R$: với LRU, một lần đọc lại trúng khi và chỉ khi từ lần
dùng trước tới giờ có ít hơn $R$ trọng số **khác** đã được chạm tới. Đếm con số đó một lần cho mỗi
lần đọc là biết số lần nạp ở mọi sức chứa.

**Dự đoán:** với tệp thanh ghi chứa 16 trọng số, thứ tự B nạp bao nhiêu lần? Và trong 120 thứ tự,
bao nhiêu thứ tự nạp mỗi trọng số đúng một lần?

In [ ]:
import itertools
from collections import OrderedDict

VONG = {"H": canh_ra, "W": canh_ra, "C": bo_loc, "Fh": F, "Fw": F}
W_loc = rng.integers(-3, 4, (bo_loc, F, F))
X_vao = rng.integers(-3, 4, (canh_vao, canh_vao))
Y_chuan = np.array([[[int((X_vao[h:h + F, w:w + F] * W_loc[c]).sum()) for w in range(canh_ra)]
                     for h in range(canh_ra)] for c in range(bo_loc)])
Wl, Xl = W_loc.tolist(), X_vao.tolist()


def chay_theo_thu_tu(thu_tu):
    """Tích chập theo thứ tự vòng lặp `thu_tu` (ngoài cùng trước); trả về đầu ra và chuỗi trọng số đã đọc."""
    Y = [[[0] * canh_ra for _ in range(canh_ra)] for _ in range(bo_loc)]
    doc = []
    vi_tri = {ten: i for i, ten in enumerate(thu_tu)}
    for chi_so in itertools.product(*(range(VONG[ten]) for ten in thu_tu)):
        h, w, c, fh, fw = (chi_so[vi_tri[ten]] for ten in ("H", "W", "C", "Fh", "Fw"))
        Y[c][h][w] += Wl[c][fh][fw] * Xl[h + fh][w + fw]
        doc.append((c * F + fh) * F + fw)
    return np.array(Y), doc


def khoang_cach_lru(doc):
    """Với mỗi lần đọc: số trọng số khác đã chạm tới từ lần dùng trước; lần đầu là vô cùng."""
    ngan_xep, kc = [], []
    for w in doc:
        try:
            i = ngan_xep.index(w)
            ngan_xep.pop(i)
        except ValueError:
            i = 10**9
        kc.append(i)
        ngan_xep.insert(0, w)
    return np.array(kc)


def lru_truc_tiep(doc, R):
    """Tệp thanh ghi LRU chạy thẳng, để đối chiếu với mẹo khoảng cách."""
    tep, nap = OrderedDict(), 0
    for w in doc:
        if w in tep:
            tep.move_to_end(w)
        else:
            nap += 1
            tep[w] = None
            if len(tep) > R:
                tep.popitem(last=False)
    return nap


cac_R = np.arange(1, 161)
nap_theo_R = {}
for thu_tu in itertools.permutations(VONG):
    Y, doc = chay_theo_thu_tu(thu_tu)
    assert np.array_equal(Y, Y_chuan)                     # cùng kết quả ở mọi thứ tự
    kc = khoang_cach_lru(doc)
    nap_theo_R[thu_tu] = np.array([(kc >= R).sum() for R in cac_R])
assert len(nap_theo_R) == 120

THU_TU_A = ("C", "Fh", "Fw", "H", "W")
THU_TU_B = ("H", "W", "C", "Fh", "Fw")
for R in (1, 9, 16, 143):
    _, doc = chay_theo_thu_tu(THU_TU_B)
    assert lru_truc_tiep(doc, R) == nap_theo_R[THU_TU_B][R - 1]
print("cả 120 thứ tự cho cùng đầu ra; mẹo khoảng cách khớp với LRU chạy thẳng")

nap_A = nap_theo_R[THU_TU_A][16 - 1]
nap_B = nap_theo_R[THU_TU_B][16 - 1]
theo_sach("thứ tự A: số lần nạp trọng số", nap_A, sach=144, nguon="36 · 16 × 9 · 144 · 36 × 144 · 36 lần")
theo_sach("thứ tự B, tệp chứa 16 trọng số", nap_B, sach=5184, nguon="5,184")
theo_sach("tỉ lệ B ÷ A", nap_B / nap_A, sach=36, nguon="36 · 16 × 9 · 144 · 36 × 144 · 36 lần")
print(f"thứ tự A nạp {nap_theo_R[THU_TU_A][0]} lần ngay cả khi tệp chỉ chứa 1 trọng số")
print(f"thứ tự B cần tệp chứa {cac_R[nap_theo_R[THU_TU_B] == 144][0]} trọng số mới về được 144 lần nạp")

In [ ]:
gia_tri = sorted({int(v) for arr in nap_theo_R.values() for v in arr})
nguong = [int(R) for i, R in enumerate(cac_R)
          if i == 0 or any(arr[i] != arr[i - 1] for arr in nap_theo_R.values())]
print("số lần nạp chỉ nhận các giá trị:", ", ".join(f"{v:,}" for v in gia_tri),
      f"(= 144 × {', '.join(str(v // 144) for v in gia_tri)})")
print("sức chứa mà ở đó phân bố thay đổi:", nguong)
assert gia_tri == [144, 6 * 144, 36 * 144]                           # ba mức, mỗi mức gấp 6 lần
assert any(arr[0] == 144 for arr in nap_theo_R.values())             # có thứ tự chỉ cần một thanh ghi
assert all(arr[144 - 1] == 144 for arr in nap_theo_R.values())       # đủ 144 thì mọi thứ tự như nhau
for R in nguong:
    dem = {v: int(sum(arr[R - 1] == v for arr in nap_theo_R.values())) for v in gia_tri}
    print(f"  R = {R:>3}: " + ", ".join(f"{dem[v]:>3} thứ tự nạp {v:,} lần" for v in gia_tri))

In [ ]:
mau_nap = dict(zip(gia_tri, ["#9ec5f0", "#2a78d6", "#123f7a"]))
fig, ax = plt.subplots(figsize=(7.5, 3.8))
day = np.zeros(len(nguong))
for v in gia_tri:
    dem = np.array([sum(arr[R - 1] == v for arr in nap_theo_R.values()) for R in nguong])
    ax.bar([str(R) for R in nguong], dem, bottom=day, color=mau_nap[v], width=0.62,
           edgecolor="white", linewidth=1.5, label=f"nạp {v:,} lần")
    day += dem
ax.set_ylim(0, 150)
ax.set_yticks([0, 30, 60, 90, 120])
ax.set_xlabel("sức chứa của tệp thanh ghi R (trọng số); giữa hai cột không có gì thay đổi")
ax.set_ylabel("số thứ tự vòng lặp")
ax.set_title("Ví dụ 1.4: 120 thứ tự, chia theo số lần nạp trọng số")
ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Ba điều hiện ra. Thứ nhất, 120 thứ tự chỉ rơi vào ba mức, mỗi mức gấp 6 lần mức trước: một vòng
không gian 6 bước nằm ngoài nhóm trọng số đang dùng là nhân số lần nạp lên 6. Thứ hai, có những
thứ tự đạt 144 lần nạp ngay với một thanh ghi, như thứ tự A: thứ tự đúng thay được cho phần cứng.
Thứ ba, khi tệp thanh ghi đủ 144 trọng số thì mọi thứ tự đều như nhau, nên một bộ tăng tốc có bộ
nhớ cục bộ lớn ít nhạy với thứ tự vòng lặp hơn.

Sách ghi rõ đây là phép đếm dưới hai giả định giới hạn. Kernel thật còn phải tính lưu lượng của giá
trị kích hoạt và tổng riêng phần, hành vi của cache và việc vectơ hoá.

### Không gian ánh xạ lớn tới đâu

Năm vòng lặp đã cho 120 thứ tự. Sách nhân các cận trên minh hoạ ("toy upper bound") để cho thấy
không gian ánh xạ lớn nhanh thế nào. Mỗi số đi với bộ tham số riêng của nó, nên không nhân chúng
với nhau được.

In [ ]:
vong_tich_chap = sach(7, nguon="7 · 5,040")
theo_sach("thứ tự của 7 vòng lặp, 7!", math.factorial(vong_tich_chap), sach=5040, nguon="7 · 5,040")
sau, ba = sach(6, nguon="6 · 3 · 120"), sach(3, nguon="6 · 3 · 120")
theo_sach("chọn 3 trong 6 vòng để song song, 6!/3!", math.factorial(sau) // math.factorial(sau - ba),
          sach=120, nguon="6 · 3 · 120")
n_lua_chon = sach(4, nguon="5 · 3 · 4 · 4^15 · 15 · 1,073,741,824")
N_tinh = sach(5, nguon="5 · 3 · 4 · 4^15 · 15 · 1,073,741,824")
N_nho = sach(3, nguon="5 · 3 · 4 · 4^15 · 15 · 1,073,741,824")
theo_sach("cách đặt dữ liệu, 4^(5·3)", n_lua_chon ** (N_tinh * N_nho), sach=1_073_741_824,
          nguon="5 · 3 · 4 · 4^15 · 15 · 1,073,741,824")
# Ví dụ 1.4 đếm được hết: 120 thứ tự × 160 sức chứa, mỗi lượt 5,184 MAC.
print(f"ở trên ta vừa chạy {len(nap_theo_R) * VONG['H'] * VONG['W'] * VONG['C'] * F * F:,} MAC để duyệt hết 120 thứ tự")

## Thử thêm

1. Ở phần 2, gọi `mang_systolic` với $M = 1$, tức một vectơ đầu vào duy nhất như suy luận batch 1,
   trên lưới 128 × 128. Mức sử dụng còn bao nhiêu? So với công thức $1/(K + N - 1)$.
2. Ở ô `scratchpad`, đổi `BO_NHO_NHAP` xuống `64 * 1024`. Khối lớn nhất còn vừa là bao nhiêu, và
   mật độ tính toán giảm mấy lần so với khi bộ nhớ nháp là 256 KiB?
3. Ở phần 4, tìm trong `nap_theo_R` những thứ tự đạt 144 lần nạp khi $R = 1$. Chúng có chung điểm
   gì về vị trí của hai vòng `H` và `W`?

## Tóm lại

* Ví dụ Tensor Core của sách đếm được từ hình dạng tensor: 268.4 triệu MAC, và 268.4 ms trên một bộ
  xử lý vô hướng ở 2 GHz; con số 0.5 ms là giả định minh hoạ.
* Một mảng systolic giữ trọng số tại chỗ cho kết quả đúng chỉ bằng thanh ghi giữa các ô, và cần
  $M + K + N - 2$ chu kỳ; mức sử dụng $M/(M + K + N - 2)$ cho thấy vì sao batch nhỏ khó lấp đầy một lưới lớn.
* Lợi thế năng lượng 232.8 lần của napkin math 1.1 phụ thuộc cách đếm: nó đo tính cục bộ so với
  một mốc cố ý ngây thơ.
* Chia khối nâng mật độ tính toán xấp xỉ $T/4$ ở FP32, và bộ nhớ nhanh, cần $3T^2$ phần tử, là trần
  của nó; chiều không chia hết cho cỡ khối thì trả thuế phần rìa.
* Cùng một tích chập, thứ tự vòng lặp đổi số lần nạp trọng số từ 144 lên 5,184, trừ khi tệp thanh
  ghi đủ lớn để chứa mọi trọng số.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tăng tốc phần cứng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch11-hw-acceleration/), dựng từ chương
[*Hardware Acceleration*](https://mlsysbook.ai/vol1/hw_acceleration/hw_acceleration.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_11_hw_accel.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.